# VMamba-T s2l5 Training
Uses the prebuilt T4 selective-scan wheel you already built. The notebook stops if CUDA selective scan is not active.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT = Path("/content/TTTN")
assert PROJECT.exists(), "Upload/unzip the project to /content/TTTN first"
assert (PROJECT / "data" / "3cad_ani").exists(), "Dataset missing at /content/TTTN/data/3cad_ani"
%cd /content/TTTN


In [ ]:
!pip install -q -r requirements/ml-kaggle.txt
!python scripts/verification/check_protocol.py


## Verify Colab runtime before installing the T4 wheel


In [ ]:
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("CC:", torch.cuda.get_device_capability(0))


## Install prebuilt Mamba CUDA wheel + official VMamba repo


In [ ]:
!PROJECT_ROOT=/content/TTTN bash scripts/setup/setup_vmamba_colab.sh


## Mandatory VMamba runtime test


In [ ]:
!python test_vmamba_runtime.py


## Main training
Micro-batch 1 × accumulation 4 = effective batch 4.


In [ ]:
!python scripts/training/train_vmamba.py \
  --epochs 50 \
  --batch-size 1 \
  --grad-accum 4 \
  --augmentation photometric \
  --run-name main_seed42


## Validation — select threshold


In [ ]:
!python scripts/evaluation/evaluate_model.py \
  --model vmamba \
  --checkpoint results/vmamba_t_s2l5/main_seed42/checkpoints/best.pt \
  --split val --warmup-batches 5


## Final Test


In [ ]:
!python scripts/evaluation/evaluate_model.py \
  --model vmamba \
  --checkpoint results/vmamba_t_s2l5/main_seed42/checkpoints/best.pt \
  --split test --warmup-batches 5


## Inspect evidence


In [ ]:
from IPython.display import display, Image
import pandas as pd
base="results/vmamba_t_s2l5/main_seed42"
for f in [
    "curves/learning_curve_loss.png",
    "curves/learning_curve_dice.png",
    "curves/loss_components.png",
    "test/figures/image_roc_curve.png",
    "test/figures/image_pr_curve.png",
    "test/figures/confusion_matrix.png",
]: display(Image(f"{base}/{f}"))
display(pd.read_csv(f"{base}/test/main_metrics.csv"))
